In [64]:
!pip install pandas
!pip install yfinance
!pip install scipy
!pip install ta

In [65]:
import pandas as pd
import yfinance as yf
import numpy as np
from scipy import stats
from datetime import datetime, timedelta
import ta

In [89]:
df = pd.read_csv('nasdaq_stocks.csv')  # Ensure the file contains a 'Symbol' column
nasdaq_stocks = df.sample(n=750, random_state=None)
print(nasdaq_stocks)

     Symbol                                      Security Name
4476   VERU                           Veru Inc. - Common Stock
4560  VSTEW                 Vast Renewables Limited - Warrants
1143  CYTHW                 Cyclo Therapeutics, Inc. - Warrant
1283   DWUS        AdvisorShares Dorsey Wright FSM US Core ETF
2181   IMNM                      Immunome, Inc. - Common Stock
...     ...                                                ...
4509  VLYPO  Valley National Bancorp - 5.5% Fixed to Floati...
3273   PCTY       Paylocity Holding Corporation - Common Stock
4532   VRAR             The Glimpse Group, Inc. - Common Stock
3871  SHOTW                        Safety Shot, Inc. - Warrant
3181   OPOF     Old Point Financial Corporation - Common Stock

[750 rows x 2 columns]


In [90]:
nasdaq_stocks

,Symbol,Security Name
4476,VERU,Veru Inc. - Common Stock
4560,VSTEW,Vast Renewables Limited - Warrants
1143,CYTHW,"Cyclo Therapeutics, Inc. - Warrant"
1283,DWUS,AdvisorShares Dorsey Wright FSM US Core ETF
2181,IMNM,"Immunome, Inc. - Common Stock"
...,...,...
4509,VLYPO,Valley National Bancorp - 5.5% Fixed to Floati...
3273,PCTY,Paylocity Holding Corporation - Common Stock
4532,VRAR,"The Glimpse Group, Inc. - Common Stock"
3871,SHOTW,"Safety Shot, Inc. - Warrant"


In [91]:
def calculate_technical_indicators_with_ta(df, symbol_column='Symbol'):
    """ 
    STEP 1: 
    Calculates the Relative Strength (RS) rating of stocks relative to the S&P 500.

    Parameters:
    - df (pd.DataFrame): DataFrame containing stock symbols.
    - symbol_column (str): Name of the column containing stock symbols.

    Calculates technical indicators for each stock using the 'ta' library.
    
    STEP 2: Aggregate Indicator Data
    Indicators:
    - 50 Day Moving Average
    - 150 Day Moving Average
    - 200 Day Moving Average
    - 200 Day Moving Average from 20 Days Ago
    - 52 Week Low
    - 52 Week High
    - Previous Close
    - Previous Open
    
    Returns:
    - pd.DataFrame: Original DataFrame with an added columns.
    """
    
    weeks_52 = 252  # Approximate number of trading days in 52 weeks
    end_date = datetime.today()
    start_date = end_date - timedelta(days=weeks_52 * 2)  # Buffer for non-trading days
    
    ###STEP 1
    benchmark_symbol = '^GSPC'  # S&P 500 index symbol in Yahoo Finance

    # Fetch benchmark data
    print("Fetching S&P 500 data...")
    benchmark_data = yf.download(benchmark_symbol, start=start_date, end=end_date, progress=False)
    if benchmark_data.empty:
        raise ValueError("Failed to fetch S&P 500 data.")
    benchmark_data = benchmark_data['Adj Close'].dropna()

    # Calculate benchmark returns
    benchmark_start = benchmark_data.iloc[0][benchmark_symbol]
    benchmark_end = benchmark_data.iloc[-1][benchmark_symbol]
    benchmark_return = (benchmark_end / benchmark_start - 1) * 100  # Percentage
    
    ###STEP 2
    ma_periods = [50, 150, 200]

    # Lists to store results
    symbols = []
    rs_returns = []
    ma_50_list = []
    ma_150_list = []
    ma_200_list = []
    ma_200_20_list = []
    low_52w_list = []
    high_52w_list = []
    curr_close_list = []

    # Iterate over each symbol to calculate indicators
    for symbol in df[symbol_column]:
        try:
            #print(f"Fetching data for {symbol} for technical indicators...")
            stock_data = yf.download(symbol, start=start_date, end=end_date, progress=False)
            
            if stock_data.empty:
                #print(f"Warning: No data fetched for {symbol}. Skipping.")
                continue
                
            adj_close = stock_data['Adj Close'].dropna()
            
            # Ensure we have enough data
            if len(adj_close) < max(ma_periods + [weeks_52]):
                #print(f"Warning: Not enough data for {symbol}. Skipping.")
                continue

            # Calculate stock returns
            stock_start = adj_close.iloc[0][symbol]
            stock_end = adj_close.iloc[-1][symbol]
            sr = (stock_end / stock_start - 1) * 100  # Percentage
            # Relative Strength
            rs = sr - benchmark_return
           
            # Calculate Moving Averages
            ma_50 = adj_close.rolling(window=50).mean().iloc[-1][symbol]
            ma_150 = adj_close.rolling(window=150).mean().iloc[-1][symbol]
            ma_200 = adj_close.rolling(window=200).mean().iloc[-1][symbol]
            ma_200_20 = adj_close.rolling(window=200).mean().iloc[-20][symbol]
            
            # Calculate 52 Week Low and High
            low_52w = adj_close.rolling(window=weeks_52).min().iloc[-1][symbol]
            high_52w = adj_close.rolling(window=weeks_52).max().iloc[-1][symbol]

            curr_close = adj_close.iloc[-1][symbol]
            
            # Append results
            symbols.append(symbol)
            ma_50_list.append(ma_50)
            ma_150_list.append(ma_150)
            ma_200_list.append(ma_200)
            ma_200_20_list.append(ma_200_20)
            low_52w_list.append(low_52w)
            high_52w_list.append(high_52w)
            curr_close_list.append(curr_close)
            rs_returns.append(rs)

            #print(f"{symbol}: 50 MA = {ma_50:.2f}, 150 MA = {ma_150:.2f}, 200 MA = {ma_200:.2f}, "
             #     f"52W Low = {low_52w:.2f}, 52W High = {high_52w:.2f}")

        except Exception as e:
            print(f"Error processing {symbol}: {e}")
            continue

    # Create a DataFrame for Indicators
    indicators_df = pd.DataFrame({
        'Stock': symbols,
        'RS_Rating': rs_returns,
        'SMA_50': ma_50_list,
        'SMA_150': ma_150_list,
        'SMA_200': ma_200_list,
        'SMA_200_20': ma_200_20_list,
        '52 Week Low': low_52w_list,
        '52 Week High': high_52w_list,
        'Current Close': curr_close_list
    })

    return indicators_df

In [92]:
nasdaq_stocks2 = calculate_technical_indicators_with_ta(nasdaq_stocks)

Fetching S&P 500 data...



1 Failed download:
['VSTEW']: YFInvalidPeriodError("%ticker%: Period 'max' is invalid, must be one of ['1d', '5d']")

1 Failed download:
['ADVWW']: YFInvalidPeriodError("%ticker%: Period 'max' is invalid, must be one of ['1d', '5d']")

1 Failed download:
['ARBEW']: YFInvalidPeriodError("%ticker%: Period 'max' is invalid, must be one of ['1d', '5d']")

1 Failed download:
['ARKOW']: YFInvalidPeriodError("%ticker%: Period 'max' is invalid, must be one of ['1d', '5d']")

1 Failed download:
['FUFUW']: YFInvalidPeriodError("%ticker%: Period 'max' is invalid, must be one of ['1d', '5d']")

1 Failed download:
['BRLSW']: YFInvalidPeriodError("%ticker%: Period 'max' is invalid, must be one of ['1d', '5d']")

1 Failed download:
['SIMAW']: YFInvalidPeriodError("%ticker%: Period 'max' is invalid, must be one of ['1d', '5d']")

1 Failed download:
['RMSGW']: YFInvalidPeriodError("%ticker%: Period 'max' is invalid, must be one of ['1d', '5d']")

1 Failed download:
['GIGGW']: YFInvalidPeriodError("%ti

In [93]:
nasdaq_stocks2

,Stock,RS_Rating,SMA_50,SMA_150,SMA_200,SMA_200_20,52 Week Low,52 Week High,Current Close
0,VERU,-78.978117,0.764014,0.905805,0.896353,0.874150,0.370000,1.790000,0.710700
1,CYTHW,-97.013396,0.157800,0.181500,0.204705,0.218605,0.080000,0.330000,0.140000
2,DWUS,-1.088080,48.055280,46.370120,45.618990,44.909165,37.873421,49.950001,49.869999
3,IMNM,41.890654,12.307000,13.616433,15.599875,16.385025,7.300000,27.340000,13.540000
4,CVGW,-53.290646,27.793650,25.744502,26.107301,26.018544,20.357491,29.433950,27.540001
...,...,...,...,...,...,...,...,...,...
607,FHTX,-41.561543,8.155000,7.067933,6.916950,6.536700,2.820000,9.990000,7.620000
608,VLYPO,-14.698011,25.195918,24.195090,23.636787,23.254539,19.545479,25.700001,25.275900
609,PCTY,-40.104174,184.762801,162.226400,163.278200,159.302275,131.850006,213.789993,207.570007
610,VRAR,-113.911569,0.695530,0.888183,0.983497,1.037790,0.581000,1.655000,0.735500


In [94]:
# List to collect rows before creating the final DataFrame
rows_list = []

# Iterate over each stock in nasdaq_stocks
for i in nasdaq_stocks2.index:
    stock = str(nasdaq_stocks2["Stock"][i])

    try:
        print(f"Fetching data for {stock}...")
        
        moving_average_50 = nasdaq_stocks2["SMA_50"][i]
        moving_average_150 = nasdaq_stocks2["SMA_150"][i]
        moving_average_200 = nasdaq_stocks2["SMA_200"][i]
        moving_average_200_20 = nasdaq_stocks2["SMA_200_20"][i]
        low_52 = nasdaq_stocks2["52 Week Low"][i]
        high_52 = nasdaq_stocks2["52 Week High"][i]
        current_close = nasdaq_stocks2["Current Close"][i]
        rs_rating = nasdaq_stocks2["RS_Rating"][i]
        
        # Define conditions
        cond_1 = current_close > moving_average_150 > moving_average_200
        cond_2 = moving_average_150 > moving_average_200
        cond_3 = moving_average_200 > moving_average_200_20
        cond_4 = moving_average_50 > moving_average_150
        cond_5 = current_close > moving_average_50
        cond_6 = current_close >= (1.3 * low_52)
        cond_7 = current_close >= (0.75 * high_52)
        cond_8 = rs_rating > 50

        # Check if all conditions are met
        if all([cond_1, cond_2, cond_3, cond_4, cond_5, cond_6, cond_7, cond_8]):
        #if all([cond_2, cond_4]):
            row = {
                'Stock': stock,
                'SMA_50': moving_average_50,
                'SMA_150': moving_average_150,
                'SMA_200': moving_average_200,
                'SMA_200_20': moving_average_200_20,
                '52 Week Low': low_52,
                '52 Week High': high_52,
                'Current Close': current_close
            }
            rows_list.append(row)
            print(f"{stock} meets all conditions. Added to final_df.")
        else:
            print(f"{stock} does not meet all conditions.")

    except Exception as e:
        print(f"Error processing {stock}: {e}")

# Create the final DataFrame from the collected rows
if rows_list:
    final_df = pd.DataFrame(rows_list)
    print("\nFinal DataFrame with Stocks Meeting All Conditions:")
    print(final_df)
else:
    print("No stocks met all conditions.")


Fetching data for VERU...
VERU does not meet all conditions.
Fetching data for CYTHW...
CYTHW does not meet all conditions.
Fetching data for DWUS...
DWUS does not meet all conditions.
Fetching data for IMNM...
IMNM does not meet all conditions.
Fetching data for CVGW...
CVGW does not meet all conditions.
Fetching data for HUDAU...
HUDAU does not meet all conditions.
Fetching data for PMTS...
PMTS does not meet all conditions.
Fetching data for DVY...
DVY does not meet all conditions.
Fetching data for UNIY...
UNIY does not meet all conditions.
Fetching data for SKIN...
SKIN does not meet all conditions.
Fetching data for LUNG...
LUNG does not meet all conditions.
Fetching data for PTEC...
PTEC does not meet all conditions.
Fetching data for PSCU...
PSCU does not meet all conditions.
Fetching data for CPTN...
CPTN does not meet all conditions.
Fetching data for PAGP...
PAGP does not meet all conditions.
Fetching data for XNCR...
XNCR does not meet all conditions.
Fetching data for PRTS

In [95]:
print(final_df)

    Stock      SMA_50     SMA_150     SMA_200  SMA_200_20  52 Week Low  \
0    HNST    4.858400    3.855200    3.794875    3.441525     2.280000   
1    JANX   49.873200   46.616666   45.076325   41.169925     7.930000   
2   LLYVA   57.996200   44.425333   43.064650   40.161300    31.480000   
3    ADSE   13.614900   12.484100   11.984425   11.438850     6.300000   
4    RKLB   13.940100    8.143000    7.123725    5.663475     3.530000   
5    RIGL   18.794800   13.403800   13.352350   12.154100     7.730000   
6    CRDO   40.105100   31.449367   28.759075   26.374325    16.920000   
7    VNOM   51.156680   43.885714   41.943302   39.717410    27.709820   
8    PRCT   84.205400   73.291666   67.329400   62.910450    38.029999   
9      TW  130.662867  117.174676  113.630897  110.433322    87.637428   
10   DJCO  517.268600  453.014066  428.647200  405.582899   313.500000   
11    NXL    2.310560    1.524220    1.418170    1.083355     0.290000   
12   DXPE   59.012200   52.826933   51

In [96]:
#### VALUE SCREENER BELOW (USE IN CONJUNCTION OR SEPERATE)

In [100]:
def fetch_financial_data(ticker):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        # Financial Metrics
        operating_margin = info.get('operatingMargins', None)
        debt_to_equity = info.get('debtToEquity', None)
        pb_ratio = info.get('priceToBook', None)
        pe_ratio = info.get('trailingPE', None)
        peg_ratio = info.get('pegRatio', None)

        ## Can't use yfinance, see other notebook for how to do this or another 3rd party thing
        insider_transactions = None

        return {
            'Ticker': ticker,
            'Operating Margin': operating_margin,
            'Debt-to-Equity': debt_to_equity,
            'P/B Ratio': pb_ratio,
            'P/E Ratio': pe_ratio,
            'PEG Ratio': peg_ratio,
        }
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return None

In [103]:
def screen_stocks(tickers):
    screened_stocks = []
    for ticker in tickers['Stock']:
        data = fetch_financial_data(ticker)
        if data:
            try:
                print(ticker)
                if (
                    data['Operating Margin'] is not None and data['Operating Margin'] > 0 and
                    #data['Debt-to-Equity'] is not None and data['Debt-to-Equity'] < 2.5 and needs to be industry specific
                    data['P/B Ratio'] is not None and data['P/B Ratio'] < 1.5 and
                    data['P/E Ratio'] is not None and data['P/E Ratio'] < 20  
                    #data['PEG Ratio'] is not None and data['PEG Ratio'] < 2
                    # Insider Transactions filter is omitted
                ):
                    screened_stocks.append(data)
                    print(f"Screened In: {ticker}")
            except TypeError:
                pass
    return screened_stocks

In [104]:
print(screen_stocks(final_df))
#print(screen_stocks("BSRR"))

BSRR
Screened In: BSRR
[{'Ticker': 'BSRR', 'Operating Margin': 0.38716, 'Debt-to-Equity': None, 'P/B Ratio': 1.2186918, 'P/E Ratio': 12.222222, 'PEG Ratio': None}]
